# Tabela do benchmark — treino por épocas × one-shot, por binarização e semente

A semente muda a divisão treino/validação/teste, o mapeamento das RAMs e o embaralhamento das épocas, então cada semente é uma repetição do experimento. Em **cada semente**, cada modelo fica com a configuração de **melhor acurácia de validação** e a tabela mostra o **teste** dela (o teste não participa da escolha). As sementes aparecem lado a lado, seguidas da **média ± desvio-padrão** entre elas (o desvio só existe com 2 ou mais sementes).

| coluna | o que é |
|---|---|
| n teste | tamanho do conjunto de teste (divisão estratificada 70/15/15, ou o teste oficial); é o mesmo em todas as sementes |
| sementes | quantas sementes o conjunto tem |
| Épocas sK | teste do treino por épocas escolhido na validação da semente K, entre bits × n-tupla × b × ρ₀ × τ (empates: correção mais fraca, depois menos bits e menor n-tupla — `summarize.choose`) |
| One-shot sK | teste do one-shot escolhido na validação da semente K, entre bits × n-tupla |
| Δ sK, z sK | Épocas − One-shot (pp) e o z de duas proporções Δ / √(p_E(1−p_E)/n + p_O(1−p_O)/n) **dentro da semente**, conservador (ignora o pareamento); \|z\| ≥ 1,96 em negrito |
| média ± dp | média e desvio-padrão amostral (ddof = 1) entre as sementes |
| Δ > 0 | em quantas sementes o treino por épocas ganhou |

**p-valor por conjunto** (uma tabela por binarização, mais a comparação lado a lado no fim): **teste t pareado entre sementes**, ou seja, o t de uma amostra sobre os Δ de cada semente (H₀: Δ médio = 0), com gl = sementes − 1. Mede se o ganho se repete quando a semente muda (divisão, mapeamento, embaralhamento). Com poucas sementes o limiar é alto (|t| ≥ 12,71 com 1 gl, 4,30 com 2, 2,78 com 4).

**Resumo** de cada binarização: com o **Δ médio de cada conjunto** (as sementes são agregadas antes, pois duas sementes do mesmo conjunto não são dois conjuntos independentes), vitórias/empates/derrotas, Δ médio e mediano, Wilcoxon (diferenças ≠ 0) e teste de sinais; e, por semente, vitórias/empates/derrotas e Δ médio.

Abaixo de cada tabela principal: o p-valor por conjunto, e validação, oráculo (maior teste do grid, só um teto) e configuração escolhida por semente.

In [1]:
import math
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import Markdown, display

# the notebook lives in experiments/, but also works when started from the repository root
HERE = Path.cwd() if (Path.cwd() / "summarize.py").exists() else Path.cwd() / "experiments"
sys.path.insert(0, str(HERE))
import datasets
import summarize as sm

RESULTS_DIR = HERE / "results"
DATA_DIR = HERE / "data"
SEEDS = None  # None: every seed with results for both models; or a list, e.g. [0, 1]
ENCODINGS = {"distributive": "Termômetro distributivo", "thermometer": "Termômetro linear"}

## Sementes disponíveis e tamanho do teste

Uma semente entra quando tem resultados dos dois modelos. O tamanho do teste segue a regra do `run_grid.split`: o teste oficial quando existe; senão, em cada classe com *k* exemplos, *k* − round(0,70·*k*) − round(0,15·*k*) (não depende da semente).

In [2]:
def test_size(name):
    ds = datasets.load(name, str(DATA_DIR))
    if ds.has_official_test:
        return len(ds.y_test)
    _, counts = np.unique(ds.y, return_counts=True)
    return int(sum(k - int(round(0.70 * k)) - int(round(0.15 * k)) for k in counts))


data = sm.load(str(RESULTS_DIR))
seeds_of = {}
for (name, encoding), seeds in sorted(data.items()):
    if encoding not in ENCODINGS:
        continue
    ok = [s for s, rows in sorted(seeds.items()) if (SEEDS is None or s in SEEDS) and len(sm.choose(rows)) == len(sm.MODELS)]
    skipped = [s for s in sorted(seeds) if s not in ok and (SEEDS is None or s in SEEDS)]
    if skipped:
        print("%s / %s: semente(s) %s fora (sem os dois modelos)" % (name, encoding, skipped))
    if ok:
        seeds_of[(name, encoding)] = ok

n_test = {name: test_size(name) for name in sorted({k[0] for k in seeds_of})}
pd.DataFrame([{"conjunto": k[0], "binarização": ENCODINGS[k[1]], "sementes": ", ".join(map(str, s)), "n teste": n_test[k[0]]}
              for k, s in sorted(seeds_of.items())])

jsc / thermometer: semente(s) [0] fora (sem os dois modelos)


,conjunto,binarização,sementes,n teste
0,australian,Termômetro distributivo,"0, 1, 2, 3",104
1,australian,Termômetro linear,"0, 1, 2, 3",104
2,blood,Termômetro distributivo,"0, 1, 2, 3",111
3,blood,Termômetro linear,"0, 1, 2, 3",111
4,christine,Termômetro distributivo,"0, 1, 2, 3",814
5,christine,Termômetro linear,"0, 1, 2, 3",814
6,fashion-mnist,Termômetro distributivo,"1, 2, 3",10000
7,fashion-mnist,Termômetro linear,"1, 2, 3",10000
8,higgs,Termômetro distributivo,"0, 1, 2, 3",14708
9,higgs,Termômetro linear,"0, 1, 2, 3",14708


## Um resultado por conjunto, binarização e semente

In [3]:
def seed_row(name, encoding, seed):
    rows = data[(name, encoding)][seed]
    chosen = sm.choose(rows)
    e_cfg, e_test, e_val = chosen["epochs + bleaching"]
    o_cfg, o_test, o_val = chosen["one-shot + bleaching"]
    pe, po, n = e_test["acc"], o_test["acc"], n_test[name]
    se = math.sqrt(pe * (1 - pe) / n + po * (1 - po) / n)
    return dict(
        dataset=name, encoding=encoding, seed=seed, n_test=n,
        epochs=100 * pe, oneshot=100 * po, delta=100 * (pe - po), delta_examples=int(round((pe - po) * n)),
        z=(pe - po) / se if se > 0 else 0.0,
        epochs_val=100 * e_val["acc"], oneshot_val=100 * o_val["acc"],
        oracle_epochs=100 * max(r["bleaching"]["test"]["acc"] for r in rows if r["kind"] == "epochs"),
        oracle_oneshot=100 * max(r["bleaching"]["test"]["acc"] for r in rows if r["kind"] == "oneshot"),
        epochs_bits=e_cfg["bits"], epochs_address=e_cfg["address"], epochs_b=e_cfg["b"],
        epochs_rho0=e_cfg["rho0"], epochs_tau=e_cfg["tau"], best_epoch=e_cfg["bestEpoch"], epochs_run=e_cfg["ran"],
        oneshot_bits=o_cfg["bits"], oneshot_address=o_cfg["address"])


long = pd.DataFrame([seed_row(name, encoding, seed) for (name, encoding), seeds in sorted(seeds_of.items()) for seed in seeds])

# mean and sample standard deviation over the seeds of each dataset and encoding (NaN with a single seed)
aggregate = long.groupby(["dataset", "encoding"]).agg(
    seeds=("seed", "count"), n_test=("n_test", "first"),
    epochs_mean=("epochs", "mean"), epochs_std=("epochs", "std"),
    oneshot_mean=("oneshot", "mean"), oneshot_std=("oneshot", "std"),
    delta_mean=("delta", "mean"), delta_std=("delta", "std"),
    wins=("delta", lambda d: int((d > 0).sum()))).reset_index()

MIN_SEEDS_FOR_T = 2  # the paired t needs 2 seeds (1 degree of freedom); 3 or more seeds make it much more useful


def paired_t(deltas):
    """paired t test over the seeds — one-sample t on Δ = epochs − one-shot: t, degrees of freedom, two-sided p"""
    d = np.asarray(deltas, dtype=float)
    k = len(d)
    if k < max(2, MIN_SEEDS_FOR_T):
        return float("nan"), k - 1, float("nan")
    if d.std(ddof=1) == 0:  # every seed with the same Δ
        return (float("nan"), k - 1, float("nan")) if d.mean() == 0 else (math.copysign(float("inf"), d.mean()), k - 1, 0.0)
    res = stats.ttest_1samp(d, 0.0)
    return float(res.statistic), k - 1, float(res.pvalue)


t_tests = (long.groupby(["dataset", "encoding"])["delta"]
           .apply(lambda d: pd.Series(paired_t(d), index=["t", "df", "p"])).unstack().reset_index())
aggregate = aggregate.merge(t_tests, on=["dataset", "encoding"])
aggregate["df"] = aggregate["df"].astype(int)
aggregate

,dataset,encoding,seeds,n_test,epochs_mean,epochs_std,oneshot_mean,oneshot_std,delta_mean,delta_std,wins,t,df,p
0,australian,distributive,4,104,86.778846,3.876085,83.653846,3.422140,3.125000,3.455747,3,1.808582,3,0.168229
1,australian,thermometer,4,104,85.096154,5.901267,84.375000,3.954796,0.721154,3.176960,3,0.453990,3,0.680668
2,blood,distributive,4,111,74.324324,3.163860,74.774775,3.370863,-0.450450,2.702703,2,-0.333333,3,0.760820
3,blood,thermometer,4,111,76.126126,1.163058,76.801802,0.450450,-0.675676,1.538581,1,-0.878310,3,0.444438
4,christine,distributive,4,814,70.423833,1.416333,68.796069,1.766082,1.627764,1.593504,3,2.043000,3,0.133660
5,christine,thermometer,4,814,69.686732,2.694080,69.901720,2.960323,-0.214988,1.004946,2,-0.427859,3,0.697609
6,fashion-mnist,distributive,3,10000,85.610000,0.065574,84.206667,0.178979,1.403333,0.133167,3,18.252665,2,0.002988
7,fashion-mnist,thermometer,3,10000,84.986667,0.310054,83.753333,0.124231,1.233333,0.285365,3,7.485831,2,0.017381
8,higgs,distributive,4,14708,67.776040,0.970483,61.135097,1.686837,6.640944,0.766841,4,17.320252,3,0.000419
9,higgs,thermometer,4,14708,62.916440,0.858803,57.744085,1.086071,5.172355,1.803577,4,5.735662,3,0.010523


In [4]:
def br(value, decimals=2, sign=False):
    """number in pt-BR (decimal comma, thousands dot); the minus sign is U+2212"""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return "—"
    text = ("{:+,.%df}" if sign else "{:,.%df}") % decimals
    text = text.format(value).replace(",", "_").replace(".", ",").replace("_", ".")
    if sign and float(value) == 0:
        text = text.lstrip("+-")
    return text.replace("-", "−")


def mean_std(mean, std, sign=False):
    return br(mean, sign=sign) if std is None or math.isnan(std) else "%s ± %s" % (br(mean, sign=sign), br(std))


def count(n, singular, plural):
    return "%d %s" % (n, singular if n == 1 else plural)


def tests(deltas):
    """wins/ties/losses, mean and median, Wilcoxon (non-zero differences) and sign test of a list of Δ (pp)"""
    d = np.asarray(deltas, dtype=float)
    w, t, l = int((d > 0).sum()), int((d == 0).sum()), int((d < 0).sum())
    nonzero = d[d != 0]
    return dict(wins=w, ties=t, losses=l, mean=d.mean(), median=np.median(d),
                wilcoxon=stats.wilcoxon(nonzero).pvalue if len(nonzero) > 1 else float("nan"),
                sign=stats.binomtest(w, w + l).pvalue if w + l > 0 else float("nan"))


def outcome(t):
    return "%s, %s e %s" % (count(t["wins"], "vitória", "vitórias"), count(t["ties"], "empate", "empates"),
                            count(t["losses"], "derrota", "derrotas"))


def markdown_table(encoding):
    rows = long[long["encoding"] == encoding]
    agg = aggregate[aggregate["encoding"] == encoding].set_index("dataset")
    seeds = sorted(rows["seed"].unique())
    header = ["Conjunto", "n teste", "sementes"]
    for s in seeds:
        header += ["Épocas s%d" % s, "One-shot s%d" % s, "Δ s%d" % s, "z s%d" % s]
    header += ["Épocas (média ± dp)", "One-shot (média ± dp)", "Δ (média ± dp)", "Δ > 0"]
    lines = ["### %s — sementes %s" % (ENCODINGS[encoding], ", ".join(map(str, seeds))), "",
             "| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for name, a in agg.iterrows():
        by_seed = rows[rows["dataset"] == name].set_index("seed")
        cells = [name, br(a["n_test"], 0), str(a["seeds"])]
        for s in seeds:
            if s not in by_seed.index:
                cells += ["—"] * 4
                continue
            r = by_seed.loc[s]
            bold = "**" if abs(r["z"]) >= 1.96 else ""
            cells += [br(r["epochs"]), br(r["oneshot"]), bold + br(r["delta"], sign=True) + bold, bold + br(r["z"]) + bold]
        cells += [mean_std(a["epochs_mean"], a["epochs_std"]), mean_std(a["oneshot_mean"], a["oneshot_std"]),
                  mean_std(a["delta_mean"], a["delta_std"], sign=True), "%d/%d" % (a["wins"], a["seeds"])]
        lines.append("| " + " | ".join(cells) + " |")

    t = tests(agg["delta_mean"])
    lines += ["", "**Resumo — Δ médio de cada conjunto (%d conjuntos):**" % len(agg), "",
              "- %s das épocas;" % outcome(t),
              "- Δ médio %s pp, Δ mediano %s pp;" % (br(t["mean"], sign=True), br(t["median"], sign=True)),
              "- Wilcoxon p = %s e teste de sinais p = %s." % (br(t["wilcoxon"], 3), br(t["sign"], 3)),
              "", "**Por semente:**", ""]
    for s in seeds:
        ts = tests(rows.loc[rows["seed"] == s, "delta"])
        lines.append("- semente %d (%d conjuntos): %s; Δ médio %s pp; Wilcoxon p = %s." % (
            s, ts["wins"] + ts["ties"] + ts["losses"], outcome(ts), br(ts["mean"], sign=True), br(ts["wilcoxon"], 3)))
    return "\n".join(lines)


def detail_table(encoding):
    """validation, oracle and chosen configuration of each seed"""
    rows = long[long["encoding"] == encoding].sort_values(["dataset", "seed"])
    return pd.DataFrame({
        "conjunto": rows["dataset"], "semente": rows["seed"],
        "épocas — val": rows["epochs_val"], "one-shot — val": rows["oneshot_val"],
        "oráculo épocas": rows["oracle_epochs"], "oráculo one-shot": rows["oracle_oneshot"],
        "épocas: bits / n-tupla": rows["epochs_bits"].astype(str) + " / " + rows["epochs_address"].astype(str),
        "épocas: b / ρ₀ / τ": rows["epochs_b"].astype(str) + " / " + rows["epochs_rho0"].astype(str) + " / " + rows["epochs_tau"].astype(str),
        "melhor época / rodadas": rows["best_epoch"].astype(str) + " / " + rows["epochs_run"].astype(str),
        "one-shot: bits / n-tupla": rows["oneshot_bits"].astype(str) + " / " + rows["oneshot_address"].astype(str),
    }).style.format(precision=2, decimal=",").hide(axis="index")

In [5]:
T_CRITICAL = {df: stats.t.ppf(0.975, df) for df in range(1, 30)}  # |t| needed for p < 0.05 (two-sided)


def p_text(p, bold=True):
    if p is None or math.isnan(p):
        return "—"
    text = "< 0,001" if p < 0.001 else br(p, 3)
    return "**%s**" % text if bold and p < 0.05 else text


def t_text(t):
    if t is None or math.isnan(t):
        return "—"
    return ("+∞" if t > 0 else "−∞") if math.isinf(t) else br(t)


def p_table(encoding):
    """paired t test over the seeds of each dataset of one encoding, sorted by p"""
    agg = aggregate[aggregate["encoding"] == encoding].sort_values(["p", "dataset"], na_position="last")
    rows = long[long["encoding"] == encoding]
    header = ["Conjunto", "sementes", "Δ por semente (pp)", "Δ (média ± dp)", "Δ > 0", "t", "gl", "t crítico (5%)", "p"]
    lines = ["#### p-valor por conjunto — %s (teste t pareado entre sementes)" % ENCODINGS[encoding], "",
             "| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for _, a in agg.iterrows():
        deltas = rows[rows["dataset"] == a["dataset"]].sort_values("seed")["delta"]
        lines.append("| " + " | ".join([
            a["dataset"], str(a["seeds"]), " · ".join(br(d, sign=True) for d in deltas),
            mean_std(a["delta_mean"], a["delta_std"], sign=True), "%d/%d" % (a["wins"], a["seeds"]),
            t_text(a["t"]), str(a["df"]) if a["df"] > 0 else "—",
            br(T_CRITICAL[a["df"]]) if a["df"] in T_CRITICAL else "—", p_text(a["p"])]) + " |")
    tested = agg[agg["p"].notna()]
    significant = tested[tested["p"] < 0.05]
    borderline = tested[(tested["p"] >= 0.05) & (tested["p"] < 0.10)]
    lines += ["", "- **%d de %d** conjuntos com p < 0,05 (%s);" % (
                  len(significant), len(tested),
                  ", ".join("%s %s" % (r["dataset"], "ganho" if r["delta_mean"] > 0 else "perda") for _, r in significant.iterrows()) or "nenhum"),
              "- %d com 0,05 ≤ p < 0,10 (%s);" % (len(borderline), ", ".join(borderline["dataset"]) or "nenhum"),
              "- t pareado = t de uma amostra sobre os Δ de cada semente, com gl = sementes − 1; "
              "com poucas sementes o limiar é alto (gl 1: 12,71; gl 2: 4,30; gl 4: 2,78)."]
    return "\n".join(lines)


def p_comparison():
    """distributive and linear thermometers side by side"""
    wide = aggregate.pivot(index="dataset", columns="encoding")
    encodings = [e for e in ENCODINGS if e in aggregate["encoding"].unique()]
    header = ["Conjunto"]
    for e in encodings:
        header += ["%s: Δ (média ± dp)" % ENCODINGS[e], "sementes", "p"]
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for name in wide.index:
        cells = [name]
        for e in encodings:
            if math.isnan(wide.loc[name, ("seeds", e)]):
                cells += ["—"] * 3
                continue
            cells += [mean_std(wide.loc[name, ("delta_mean", e)], wide.loc[name, ("delta_std", e)], sign=True),
                      "%d" % wide.loc[name, ("seeds", e)], p_text(wide.loc[name, ("p", e)])]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

## Termômetro distributivo

In [6]:
display(Markdown(markdown_table("distributive")))

### Termômetro distributivo — sementes 0, 1, 2, 3

| Conjunto | n teste | sementes | Épocas s0 | One-shot s0 | Δ s0 | z s0 | Épocas s1 | One-shot s1 | Δ s1 | z s1 | Épocas s2 | One-shot s2 | Δ s2 | z s2 | Épocas s3 | One-shot s3 | Δ s3 | z s3 | Épocas (média ± dp) | One-shot (média ± dp) | Δ (média ± dp) | Δ > 0 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| australian | 104 | 4 | 83,65 | 79,81 | +3,85 | 0,72 | 86,54 | 81,73 | +4,81 | 0,95 | 92,31 | 86,54 | +5,77 | 1,36 | 84,62 | 86,54 | −1,92 | −0,39 | 86,78 ± 3,88 | 83,65 ± 3,42 | +3,12 ± 3,46 | 3/4 |
| blood | 111 | 4 | 72,07 | 71,17 | +0,90 | 0,15 | 71,17 | 73,87 | −2,70 | −0,45 | 76,58 | 79,28 | −2,70 | −0,49 | 77,48 | 74,77 | +2,70 | 0,47 | 74,32 ± 3,16 | 74,77 ± 3,37 | −0,45 ± 2,70 | 2/4 |
| christine | 814 | 4 | 69,29 | 69,90 | −0,61 | −0,27 | 69,16 | 66,46 | +2,70 | 1,17 | 71,99 | 70,39 | +1,60 | 0,71 | 71,25 | 68,43 | +2,83 | 1,24 | 70,42 ± 1,42 | 68,80 ± 1,77 | +1,63 ± 1,59 | 3/4 |
| fashion-mnist | 10.000 | 3 | — | — | — | — | 85,68 | 84,31 | **+1,37** | **2,71** | 85,60 | 84,31 | **+1,29** | **2,55** | 85,55 | 84,00 | **+1,55** | **3,05** | 85,61 ± 0,07 | 84,21 ± 0,18 | +1,40 ± 0,13 | 3/3 |
| higgs | 14.708 | 4 | 67,39 | 60,06 | **+7,33** | **13,11** | 68,43 | 62,06 | **+6,36** | **11,49** | 68,70 | 63,01 | **+5,68** | **10,30** | 66,59 | 59,40 | **+7,19** | **12,80** | 67,78 ± 0,97 | 61,14 ± 1,69 | +6,64 ± 0,77 | 4/4 |
| iris | 21 | 4 | 100,00 | 100,00 | 0,00 | 0,00 | 80,95 | 76,19 | +4,76 | 0,38 | 100,00 | 100,00 | 0,00 | 0,00 | 90,48 | 90,48 | 0,00 | 0,00 | 92,86 ± 9,12 | 91,67 ± 11,25 | +1,19 ± 2,38 | 1/4 |
| jasmine | 448 | 4 | 77,01 | 78,79 | −1,79 | −0,64 | 77,90 | 76,34 | +1,56 | 0,56 | 77,46 | 77,90 | −0,45 | −0,16 | 79,02 | 76,12 | +2,90 | 1,04 | 77,85 ± 0,86 | 77,29 ± 1,28 | +0,56 ± 2,08 | 2/4 |
| jsc | 124.501 | 3 | — | — | — | — | 72,01 | 71,45 | **+0,56** | **3,10** | 71,49 | 71,14 | +0,35 | 1,91 | 71,75 | 71,43 | +0,32 | 1,75 | 71,75 ± 0,26 | 71,34 ± 0,17 | +0,41 ± 0,13 | 3/3 |
| miniboone | 19.510 | 4 | 91,89 | 91,30 | **+0,59** | **2,10** | 91,50 | 90,90 | **+0,59** | **2,07** | 92,78 | 91,96 | **+0,83** | **3,07** | 92,18 | 91,75 | +0,43 | 1,55 | 92,09 ± 0,54 | 91,48 ± 0,47 | +0,61 ± 0,16 | 4/4 |
| mnist | 10.000 | 3 | — | — | — | — | 96,11 | 95,62 | +0,49 | 1,74 | 96,33 | 95,91 | +0,42 | 1,54 | 96,14 | 95,91 | +0,23 | 0,83 | 96,19 ± 0,12 | 95,81 ± 0,17 | +0,38 ± 0,13 | 3/3 |
| nomao | 5.169 | 4 | 95,03 | 94,37 | +0,66 | 1,49 | 95,26 | 94,89 | +0,37 | 0,86 | 95,16 | 94,97 | +0,19 | 0,45 | 95,57 | 95,32 | +0,25 | 0,61 | 95,26 ± 0,23 | 94,89 ± 0,39 | +0,37 ± 0,21 | 4/4 |
| phoneme | 810 | 4 | 86,42 | 85,68 | +0,74 | 0,43 | 88,52 | 88,15 | +0,37 | 0,23 | 89,38 | 89,14 | +0,25 | 0,16 | 87,16 | 87,16 | 0,00 | 0,00 | 87,87 ± 1,33 | 87,53 ± 1,47 | +0,34 ± 0,31 | 3/4 |
| segment | 346 | 4 | 99,42 | 99,42 | 0,00 | 0,00 | 98,55 | 98,55 | 0,00 | 0,00 | 98,84 | 99,42 | −0,58 | −0,82 | 99,42 | 99,42 | 0,00 | 0,00 | 99,06 ± 0,43 | 99,21 ± 0,43 | −0,14 ± 0,29 | 0/4 |
| skin-seg | 36.758 | 4 | 99,28 | 99,26 | +0,03 | 0,43 | 99,87 | 99,82 | +0,04 | 1,50 | 99,81 | 99,73 | **+0,08** | **2,22** | 99,86 | 99,69 | **+0,16** | **4,66** | 99,70 ± 0,28 | 99,63 ± 0,25 | +0,08 ± 0,06 | 4/4 |
| sylvine | 770 | 4 | 92,73 | 87,79 | **+4,94** | **3,28** | 92,60 | 87,92 | **+4,68** | **3,10** | 90,65 | 88,70 | +1,95 | 1,26 | 90,26 | 88,96 | +1,30 | 0,84 | 91,56 ± 1,29 | 88,34 ± 0,57 | +3,21 ± 1,86 | 4/4 |

**Resumo — Δ médio de cada conjunto (15 conjuntos):**

- 13 vitórias, 0 empates e 2 derrotas das épocas;
- Δ médio +1,29 pp, Δ mediano +0,56 pp;
- Wilcoxon p = 0,002 e teste de sinais p = 0,007.

**Por semente:**

- semente 0 (12 conjuntos): 8 vitórias, 2 empates e 2 derrotas; Δ médio +1,39 pp; Wilcoxon p = 0,084.
- semente 1 (15 conjuntos): 13 vitórias, 1 empate e 1 derrota; Δ médio +1,73 pp; Wilcoxon p = 0,007.
- semente 2 (15 conjuntos): 11 vitórias, 1 empate e 3 derrotas; Δ médio +0,98 pp; Wilcoxon p = 0,091.
- semente 3 (15 conjuntos): 11 vitórias, 3 empates e 1 derrota; Δ médio +1,20 pp; Wilcoxon p = 0,012.

In [7]:
display(Markdown(p_table("distributive")))

#### p-valor por conjunto — Termômetro distributivo (teste t pareado entre sementes)

| Conjunto | sementes | Δ por semente (pp) | Δ (média ± dp) | Δ > 0 | t | gl | t crítico (5%) | p |
|---|---|---|---|---|---|---|---|---|
| higgs | 4 | +7,33 · +6,36 · +5,68 · +7,19 | +6,64 ± 0,77 | 4/4 | 17,32 | 3 | 3,18 | **< 0,001** |
| fashion-mnist | 3 | +1,37 · +1,29 · +1,55 | +1,40 ± 0,13 | 3/3 | 18,25 | 2 | 4,30 | **0,003** |
| miniboone | 4 | +0,59 · +0,59 · +0,83 · +0,43 | +0,61 ± 0,16 | 4/4 | 7,41 | 3 | 3,18 | **0,005** |
| jsc | 3 | +0,56 · +0,35 · +0,32 | +0,41 ± 0,13 | 3/3 | 5,31 | 2 | 4,30 | **0,034** |
| nomao | 4 | +0,66 · +0,37 · +0,19 · +0,25 | +0,37 ± 0,21 | 4/4 | 3,56 | 3 | 3,18 | **0,038** |
| mnist | 3 | +0,49 · +0,42 · +0,23 | +0,38 ± 0,13 | 3/3 | 4,89 | 2 | 4,30 | **0,039** |
| sylvine | 4 | +4,94 · +4,68 · +1,95 · +1,30 | +3,21 ± 1,86 | 4/4 | 3,46 | 3 | 3,18 | **0,041** |
| skin-seg | 4 | +0,03 · +0,04 · +0,08 · +0,16 | +0,08 ± 0,06 | 4/4 | 2,58 | 3 | 3,18 | 0,082 |
| phoneme | 4 | +0,74 · +0,37 · +0,25 · 0,00 | +0,34 ± 0,31 | 3/4 | 2,20 | 3 | 3,18 | 0,115 |
| christine | 4 | −0,61 · +2,70 · +1,60 · +2,83 | +1,63 ± 1,59 | 3/4 | 2,04 | 3 | 3,18 | 0,134 |
| australian | 4 | +3,85 · +4,81 · +5,77 · −1,92 | +3,12 ± 3,46 | 3/4 | 1,81 | 3 | 3,18 | 0,168 |
| iris | 4 | 0,00 · +4,76 · 0,00 · 0,00 | +1,19 ± 2,38 | 1/4 | 1,00 | 3 | 3,18 | 0,391 |
| segment | 4 | 0,00 · 0,00 · −0,58 · 0,00 | −0,14 ± 0,29 | 0/4 | −1,00 | 3 | 3,18 | 0,391 |
| jasmine | 4 | −1,79 · +1,56 · −0,45 · +2,90 | +0,56 ± 2,08 | 2/4 | 0,54 | 3 | 3,18 | 0,629 |
| blood | 4 | +0,90 · −2,70 · −2,70 · +2,70 | −0,45 ± 2,70 | 2/4 | −0,33 | 3 | 3,18 | 0,761 |

- **7 de 15** conjuntos com p < 0,05 (higgs ganho, fashion-mnist ganho, miniboone ganho, jsc ganho, nomao ganho, mnist ganho, sylvine ganho);
- 1 com 0,05 ≤ p < 0,10 (skin-seg);
- t pareado = t de uma amostra sobre os Δ de cada semente, com gl = sementes − 1; com poucas sementes o limiar é alto (gl 1: 12,71; gl 2: 4,30; gl 4: 2,78).

In [8]:
detail_table("distributive")

conjunto,semente,épocas — val,one-shot — val,oráculo épocas,oráculo one-shot,épocas: bits / n-tupla,épocas: b / ρ₀ / τ,melhor época / rodadas,one-shot: bits / n-tupla
australian,0,"88,35","86,41","86,54","86,54",4 / 4,1 / 0.8 / 0.05,2 / 6,8 / 12
australian,1,"87,38","85,44","90,38","86,54",2 / 2,1 / 0.9 / 0.05,6 / 10,16 / 8
australian,2,"85,44","80,58","94,23","89,42",2 / 2,1 / 0.1 / 0.1,0 / 4,8 / 16
australian,3,"87,38","84,47","90,38","86,54",2 / 4,2 / 0.3 / 0.0,2 / 6,2 / 8
blood,0,"82,30","77,88","80,18","78,38",4 / 8,2 / 0.9 / 0.05,6 / 10,16 / 16
blood,1,"87,61","84,07","79,28","77,48",16 / 16,2 / 0.7 / 0.05,7 / 11,16 / 16
blood,2,"84,07","79,65","85,59","83,78",32 / 2,2 / 0.9 / 0.1,7 / 11,4 / 8
blood,3,"83,19","78,76","81,08","79,28",4 / 2,1 / 0.3 / 0.05,2 / 6,32 / 4
christine,0,"72,78","70,81","72,11","69,90",16 / 16,1 / 0.9 / 0.0,0 / 4,8 / 12
christine,1,"71,92","69,33","70,02","67,08",2 / 4,2 / 0.7 / 0.05,8 / 12,32 / 16


## Termômetro linear

In [9]:
display(Markdown(markdown_table("thermometer")))

### Termômetro linear — sementes 0, 1, 2, 3

| Conjunto | n teste | sementes | Épocas s0 | One-shot s0 | Δ s0 | z s0 | Épocas s1 | One-shot s1 | Δ s1 | z s1 | Épocas s2 | One-shot s2 | Δ s2 | z s2 | Épocas s3 | One-shot s3 | Δ s3 | z s3 | Épocas (média ± dp) | One-shot (média ± dp) | Δ (média ± dp) | Δ > 0 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| australian | 104 | 4 | 79,81 | 83,65 | −3,85 | −0,72 | 80,77 | 79,81 | +0,96 | 0,17 | 92,31 | 89,42 | +2,88 | 0,72 | 87,50 | 84,62 | +2,88 | 0,60 | 85,10 ± 5,90 | 84,38 ± 3,95 | +0,72 ± 3,18 | 3/4 |
| blood | 111 | 4 | 74,77 | 77,48 | −2,70 | −0,47 | 75,68 | 76,58 | −0,90 | −0,16 | 76,58 | 76,58 | 0,00 | 0,00 | 77,48 | 76,58 | +0,90 | 0,16 | 76,13 ± 1,16 | 76,80 ± 0,45 | −0,68 ± 1,54 | 1/4 |
| christine | 814 | 4 | 68,30 | 68,80 | −0,49 | −0,21 | 66,58 | 66,34 | +0,25 | 0,10 | 71,62 | 73,10 | −1,47 | −0,67 | 72,24 | 71,38 | +0,86 | 0,39 | 69,69 ± 2,69 | 69,90 ± 2,96 | −0,21 ± 1,00 | 2/4 |
| fashion-mnist | 10.000 | 3 | — | — | — | — | 85,34 | 83,83 | **+1,51** | **2,96** | 84,86 | 83,61 | **+1,25** | **2,43** | 84,76 | 83,82 | +0,94 | 1,83 | 84,99 ± 0,31 | 83,75 ± 0,12 | +1,23 ± 0,29 | 3/3 |
| higgs | 14.708 | 4 | 62,12 | 59,34 | **+2,77** | **4,87** | 62,39 | 57,27 | **+5,12** | **8,97** | 64,03 | 56,93 | **+7,10** | **12,50** | 63,13 | 57,44 | **+5,69** | **9,99** | 62,92 ± 0,86 | 57,74 ± 1,09 | +5,17 ± 1,80 | 4/4 |
| iris | 21 | 4 | 95,24 | 95,24 | 0,00 | 0,00 | 95,24 | 95,24 | 0,00 | 0,00 | 100,00 | 95,24 | +4,76 | 1,02 | 95,24 | 95,24 | 0,00 | 0,00 | 96,43 ± 2,38 | 95,24 ± 0,00 | +1,19 ± 2,38 | 1/4 |
| jasmine | 448 | 4 | 73,88 | 73,44 | +0,45 | 0,15 | 78,35 | 75,00 | +3,35 | 1,19 | 79,02 | 71,43 | **+7,59** | **2,64** | 79,24 | 76,12 | +3,12 | 1,12 | 77,62 ± 2,52 | 74,00 ± 2,03 | +3,63 ± 2,95 | 4/4 |
| jsc | 124.501 | 3 | — | — | — | — | 70,95 | 68,16 | **+2,80** | **15,17** | 69,86 | 69,09 | **+0,77** | **4,18** | 70,49 | 67,99 | **+2,50** | **13,50** | 70,43 ± 0,55 | 68,41 ± 0,59 | +2,02 ± 1,09 | 3/3 |
| miniboone | 19.510 | 4 | 85,07 | 83,32 | **+1,76** | **4,76** | 86,59 | 83,20 | **+3,39** | **9,36** | 86,24 | 82,29 | **+3,96** | **10,75** | 86,03 | 82,61 | **+3,42** | **9,30** | 85,99 ± 0,65 | 82,85 ± 0,49 | +3,13 ± 0,95 | 4/4 |
| mnist | 10.000 | 3 | — | — | — | — | 96,30 | 96,02 | +0,28 | 1,03 | 96,38 | 95,89 | +0,49 | 1,80 | 96,34 | 95,99 | +0,35 | 1,29 | 96,34 ± 0,04 | 95,97 ± 0,07 | +0,37 ± 0,11 | 3/3 |
| nomao | 5.169 | 4 | 94,83 | 94,47 | +0,37 | 0,83 | 95,63 | 95,28 | +0,35 | 0,85 | 95,67 | 95,47 | +0,19 | 0,48 | 95,74 | 95,65 | +0,10 | 0,24 | 95,47 ± 0,43 | 95,22 ± 0,52 | +0,25 ± 0,13 | 4/4 |
| phoneme | 810 | 4 | 83,09 | 81,98 | +1,11 | 0,59 | 85,80 | 85,19 | +0,62 | 0,35 | 87,28 | 86,30 | +0,99 | 0,59 | 84,81 | 85,31 | −0,49 | −0,28 | 85,25 ± 1,76 | 84,69 ± 1,88 | +0,56 ± 0,73 | 3/4 |
| segment | 346 | 4 | 98,84 | 98,84 | 0,00 | 0,00 | 98,27 | 98,27 | 0,00 | 0,00 | 98,84 | 98,84 | 0,00 | 0,00 | 99,71 | 99,71 | 0,00 | 0,00 | 98,92 ± 0,60 | 98,92 ± 0,60 | 0,00 ± 0,00 | 0/4 |
| skin-seg | 36.758 | 4 | 99,62 | 99,50 | **+0,13** | **2,56** | 99,79 | 99,73 | +0,06 | 1,58 | 99,76 | 99,68 | +0,08 | 1,95 | 99,86 | 99,80 | **+0,06** | **2,04** | 99,76 ± 0,10 | 99,68 ± 0,13 | +0,08 ± 0,03 | 4/4 |
| sylvine | 770 | 4 | 90,00 | 80,91 | **+9,09** | **5,10** | 90,91 | 87,40 | **+3,51** | **2,22** | 92,21 | 87,53 | **+4,68** | **3,05** | 91,95 | 88,31 | **+3,64** | **2,40** | 91,27 ± 1,01 | 86,04 ± 3,44 | +5,23 ± 2,63 | 4/4 |

**Resumo — Δ médio de cada conjunto (15 conjuntos):**

- 12 vitórias, 1 empate e 2 derrotas das épocas;
- Δ médio +1,51 pp, Δ mediano +0,72 pp;
- Wilcoxon p = 0,003 e teste de sinais p = 0,013.

**Por semente:**

- semente 0 (12 conjuntos): 7 vitórias, 2 empates e 3 derrotas; Δ médio +0,72 pp; Wilcoxon p = 0,492.
- semente 1 (15 conjuntos): 12 vitórias, 2 empates e 1 derrota; Δ médio +1,42 pp; Wilcoxon p = 0,003.
- semente 2 (15 conjuntos): 12 vitórias, 2 empates e 1 derrota; Δ médio +2,22 pp; Wilcoxon p = 0,005.
- semente 3 (15 conjuntos): 12 vitórias, 2 empates e 1 derrota; Δ médio +1,60 pp; Wilcoxon p = 0,002.

In [10]:
display(Markdown(p_table("thermometer")))

#### p-valor por conjunto — Termômetro linear (teste t pareado entre sementes)

| Conjunto | sementes | Δ por semente (pp) | Δ (média ± dp) | Δ > 0 | t | gl | t crítico (5%) | p |
|---|---|---|---|---|---|---|---|---|
| miniboone | 4 | +1,76 · +3,39 · +3,96 · +3,42 | +3,13 ± 0,95 | 4/4 | 6,58 | 3 | 3,18 | **0,007** |
| higgs | 4 | +2,77 · +5,12 · +7,10 · +5,69 | +5,17 ± 1,80 | 4/4 | 5,74 | 3 | 3,18 | **0,011** |
| skin-seg | 4 | +0,13 · +0,06 · +0,08 · +0,06 | +0,08 ± 0,03 | 4/4 | 5,18 | 3 | 3,18 | **0,014** |
| fashion-mnist | 3 | +1,51 · +1,25 · +0,94 | +1,23 ± 0,29 | 3/3 | 7,49 | 2 | 4,30 | **0,017** |
| mnist | 3 | +0,28 · +0,49 · +0,35 | +0,37 ± 0,11 | 3/3 | 6,05 | 2 | 4,30 | **0,026** |
| sylvine | 4 | +9,09 · +3,51 · +4,68 · +3,64 | +5,23 ± 2,63 | 4/4 | 3,98 | 3 | 3,18 | **0,028** |
| nomao | 4 | +0,37 · +0,35 · +0,19 · +0,10 | +0,25 ± 0,13 | 4/4 | 3,89 | 3 | 3,18 | **0,030** |
| jsc | 3 | +2,80 · +0,77 · +2,50 | +2,02 ± 1,09 | 3/3 | 3,20 | 2 | 4,30 | 0,085 |
| jasmine | 4 | +0,45 · +3,35 · +7,59 · +3,12 | +3,63 ± 2,95 | 4/4 | 2,46 | 3 | 3,18 | 0,091 |
| phoneme | 4 | +1,11 · +0,62 · +0,99 · −0,49 | +0,56 ± 0,73 | 3/4 | 1,52 | 3 | 3,18 | 0,226 |
| iris | 4 | 0,00 · 0,00 · +4,76 · 0,00 | +1,19 ± 2,38 | 1/4 | 1,00 | 3 | 3,18 | 0,391 |
| blood | 4 | −2,70 · −0,90 · 0,00 · +0,90 | −0,68 ± 1,54 | 1/4 | −0,88 | 3 | 3,18 | 0,444 |
| australian | 4 | −3,85 · +0,96 · +2,88 · +2,88 | +0,72 ± 3,18 | 3/4 | 0,45 | 3 | 3,18 | 0,681 |
| christine | 4 | −0,49 · +0,25 · −1,47 · +0,86 | −0,21 ± 1,00 | 2/4 | −0,43 | 3 | 3,18 | 0,698 |
| segment | 4 | 0,00 · 0,00 · 0,00 · 0,00 | 0,00 ± 0,00 | 0/4 | — | 3 | 3,18 | — |

- **7 de 14** conjuntos com p < 0,05 (miniboone ganho, higgs ganho, skin-seg ganho, fashion-mnist ganho, mnist ganho, sylvine ganho, nomao ganho);
- 2 com 0,05 ≤ p < 0,10 (jsc, jasmine);
- t pareado = t de uma amostra sobre os Δ de cada semente, com gl = sementes − 1; com poucas sementes o limiar é alto (gl 1: 12,71; gl 2: 4,30; gl 4: 2,78).

In [11]:
detail_table("thermometer")

conjunto,semente,épocas — val,one-shot — val,oráculo épocas,oráculo one-shot,épocas: bits / n-tupla,épocas: b / ρ₀ / τ,melhor época / rodadas,one-shot: bits / n-tupla
australian,0,"86,41","82,52","90,38","88,46",4 / 4,2 / 0.7 / 0.3,2 / 6,4 / 16
australian,1,"88,35","84,47","92,31","89,42",16 / 16,1 / 0.7 / 0.0,0 / 4,16 / 16
australian,2,"86,41","83,50","94,23","90,38",2 / 2,1 / 0.3 / 0.1,3 / 7,4 / 16
australian,3,"85,44","80,58","91,35","84,62",2 / 4,2 / 0.3 / 0.0,2 / 6,2 / 8
blood,0,"84,96","81,42","81,08","78,38",16 / 12,2 / 0.7 / 0.5,5 / 9,16 / 20
blood,1,"84,07","81,42","80,18","80,18",32 / 4,2 / 0.3 / 0.2,0 / 4,8 / 16
blood,2,"84,96","83,19","81,98","81,08",16 / 8,1 / 0.1 / 0.0,2 / 6,16 / 16
blood,3,"83,19","81,42","79,28","77,48",32 / 32,2 / 0.5 / 0.05,5 / 9,32 / 16
christine,0,"71,55","70,57","70,39","69,90",16 / 24,1 / 0.7 / 0.1,0 / 4,16 / 24
christine,1,"71,43","70,94","68,06","66,83",32 / 24,1 / 0.5 / 0.0,0 / 4,32 / 24


## p-valor das duas binarizações lado a lado

Mesmo teste t pareado entre sementes das tabelas acima; p < 0,05 em negrito.

In [12]:
display(Markdown(p_comparison()))

| Conjunto | Termômetro distributivo: Δ (média ± dp) | sementes | p | Termômetro linear: Δ (média ± dp) | sementes | p |
|---|---|---|---|---|---|---|
| australian | +3,12 ± 3,46 | 4 | 0,168 | +0,72 ± 3,18 | 4 | 0,681 |
| blood | −0,45 ± 2,70 | 4 | 0,761 | −0,68 ± 1,54 | 4 | 0,444 |
| christine | +1,63 ± 1,59 | 4 | 0,134 | −0,21 ± 1,00 | 4 | 0,698 |
| fashion-mnist | +1,40 ± 0,13 | 3 | **0,003** | +1,23 ± 0,29 | 3 | **0,017** |
| higgs | +6,64 ± 0,77 | 4 | **< 0,001** | +5,17 ± 1,80 | 4 | **0,011** |
| iris | +1,19 ± 2,38 | 4 | 0,391 | +1,19 ± 2,38 | 4 | 0,391 |
| jasmine | +0,56 ± 2,08 | 4 | 0,629 | +3,63 ± 2,95 | 4 | 0,091 |
| jsc | +0,41 ± 0,13 | 3 | **0,034** | +2,02 ± 1,09 | 3 | 0,085 |
| miniboone | +0,61 ± 0,16 | 4 | **0,005** | +3,13 ± 0,95 | 4 | **0,007** |
| mnist | +0,38 ± 0,13 | 3 | **0,039** | +0,37 ± 0,11 | 3 | **0,026** |
| nomao | +0,37 ± 0,21 | 4 | **0,038** | +0,25 ± 0,13 | 4 | **0,030** |
| phoneme | +0,34 ± 0,31 | 4 | 0,115 | +0,56 ± 0,73 | 4 | 0,226 |
| segment | −0,14 ± 0,29 | 4 | 0,391 | 0,00 ± 0,00 | 4 | — |
| skin-seg | +0,08 ± 0,06 | 4 | 0,082 | +0,08 ± 0,03 | 4 | **0,014** |
| sylvine | +3,21 ± 1,86 | 4 | **0,041** | +5,23 ± 2,63 | 4 | **0,028** |

## Bleaching iterativo × DAB × rDAB

`run_grid.py --valModes DAB rDAB` repete o grid trocando a política de bleaching: ela vale na validação de cada época e na classificação de validação e teste dos dois modelos. Em **cada semente e modo**, cada modelo fica com a configuração de **melhor acurácia de validação** (`summarize.choose`, mesmos empates) e a tabela mostra o **teste** dela, em média ± dp entre as sementes; em negrito, o melhor dos três modos de cada modelo.

Entram só as sementes em que os três modos têm os dois modelos; as que faltam são listadas abaixo. **Os resultados de DAB e rDAB ainda estão incompletos.**

Os grids não são iguais: o iterativo tem b ∈ {1, 2} nas sementes 1–3 e um grid antigo na semente 0, enquanto DAB e rDAB têm só b = 1. A coluna *configurações* do detalhe mostra quantas configurações cada modo teve. Com `COMMON_GRID = True`, cada semente usa só as configurações que os três modos rodaram.

`modes_long` guarda uma linha por conjunto, binarização, semente, modelo e modo (teste, validação e configuração escolhida).

In [13]:
MODES = {"bleaching": "Iterativo", "DAB": "DAB", "rDAB": "rDAB"}  # valMode of run_grid.py -> name in the tables
MODE_ORDER = list(MODES.values())
MODEL_SHORT = {"one-shot + bleaching": "One-shot", "epochs + bleaching": "Épocas"}
COMMON_GRID = False  # True: in each seed, only the configurations that the three modes ran

by_mode = {mode: sm.load(str(RESULTS_DIR), mode) for mode in MODES}


def config_key(r):
    return (r["kind"], r["bits"], r["address"], r["b"], r["rho0"], r["tau"])


def config_text(model, cfg):
    text = "%d / %d" % (cfg["bits"], cfg["address"])
    if model == "epochs + bleaching":
        text += " · b %d · ρ₀ %s · τ %s · época %d/%d" % (cfg["b"], cfg["rho0"], cfg["tau"], cfg["bestEpoch"], cfg["ran"])
    return text


# one row per dataset, encoding, seed, model and mode: the configuration with the best validation accuracy
mode_records, mode_missing = [], []
for key in sorted(by_mode["DAB"].keys() | by_mode["rDAB"].keys()):
    name, encoding = key
    if encoding not in ENCODINGS:
        continue
    for seed in sorted(set(by_mode["DAB"].get(key, {})) | set(by_mode["rDAB"].get(key, {}))):
        if SEEDS is not None and seed not in SEEDS:
            continue
        rows = {m: by_mode[m].get(key, {}).get(seed, []) for m in MODES}
        if COMMON_GRID:
            common = set.intersection(*({config_key(r) for r in rs} for rs in rows.values()))
            rows = {m: [r for r in rs if config_key(r) in common] for m, rs in rows.items()}
        chosen = {m: sm.choose(rs) for m, rs in rows.items()}
        lacking = [MODES[m] for m in MODES if len(chosen[m]) < len(sm.MODELS)]
        if lacking:
            mode_missing.append("%s / %s, semente %d: falta %s com os dois modelos" % (
                name, ENCODINGS[encoding], seed, " e ".join(lacking)))
            continue
        for m in MODES:
            for model, short in MODEL_SHORT.items():
                cfg, test, val = chosen[m][model]
                kind = "epochs" if model == "epochs + bleaching" else "oneshot"
                mode_records.append(dict(
                    dataset=name, encoding=encoding, seed=seed, model=short, mode=MODES[m],
                    test=100 * test["acc"], val=100 * val["acc"],
                    configs=sum(r["kind"] == kind for r in rows[m]), config=config_text(model, cfg)))

modes_long = pd.DataFrame(mode_records, columns=["dataset", "encoding", "seed", "model", "mode", "test", "val", "configs", "config"])
print("Fora da comparação:\n" + "\n".join(mode_missing) if mode_missing else "Todas as sementes com DAB/rDAB entram.")


def modes_table(encoding):
    """mean ± std of the test over the seeds, per dataset, model and mode; best mode of each model in bold"""
    rows = modes_long[modes_long["encoding"] == encoding]
    if rows.empty:
        return "_Sem sementes com os três modos em %s._" % ENCODINGS[encoding]
    agg = rows.groupby(["dataset", "model", "mode"])["test"].agg(["mean", "std"])
    header = ["Conjunto", "sementes"] + ["%s %s" % (model, mode) for model in ("One-shot", "Épocas") for mode in MODE_ORDER]
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for name in sorted(rows["dataset"].unique()):
        cells = [name, ", ".join(map(str, sorted(rows.loc[rows["dataset"] == name, "seed"].unique())))]
        for model in ("One-shot", "Épocas"):
            best = max(agg.loc[(name, model, mode), "mean"] for mode in MODE_ORDER)
            for mode in MODE_ORDER:
                mean, std = agg.loc[(name, model, mode)]
                text = mean_std(mean, std)
                cells.append("**%s**" % text if np.isclose(mean, best) else text)
        lines.append("| " + " | ".join(cells) + " |")

    per_dataset = rows.groupby(["dataset", "model", "mode"])["test"].mean().unstack("mode")
    n = rows["dataset"].nunique()
    lines += ["", "**Resumo — teste médio de cada conjunto (%d conjuntos); vitórias do primeiro modo:**" % n, ""]
    for model in ("One-shot", "Épocas"):
        d = per_dataset.xs(model, level="model")
        for a, b in (("DAB", "Iterativo"), ("rDAB", "Iterativo"), ("rDAB", "DAB")):
            t = tests(d[a] - d[b])
            lines.append("- %s, %s − %s: %s; Δ médio %s pp, mediano %s pp; Wilcoxon p = %s." % (
                model, a, b, outcome(t), br(t["mean"], sign=True), br(t["median"], sign=True), p_text(t["wilcoxon"], bold=False)))
    return "\n".join(lines)


def modes_detail(encoding):
    """per seed and model: test of each mode, how many configurations each mode ran and the chosen one"""
    rows = modes_long[modes_long["encoding"] == encoding]
    if rows.empty:
        return None
    out = []
    for (name, seed, model), g in rows.groupby(["dataset", "seed", "model"]):
        g = g.set_index("mode")
        rec = {"conjunto": name, "semente": seed, "modelo": model}
        rec.update({"teste " + mode: g.loc[mode, "test"] for mode in MODE_ORDER})
        rec["configurações (it / DAB / rDAB)"] = " / ".join(str(g.loc[mode, "configs"]) for mode in MODE_ORDER)
        rec.update({"escolhida " + mode: g.loc[mode, "config"] for mode in MODE_ORDER})
        out.append(rec)
    return pd.DataFrame(out).style.format(precision=2, decimal=",").hide(axis="index")

Fora da comparação:
fashion-mnist / Termômetro distributivo, semente 2: falta DAB e rDAB com os dois modelos


### Termômetro distributivo — iterativo × DAB × rDAB

In [14]:
display(Markdown(modes_table("distributive")))
modes_detail("distributive")

| Conjunto | sementes | One-shot Iterativo | One-shot DAB | One-shot rDAB | Épocas Iterativo | Épocas DAB | Épocas rDAB |
|---|---|---|---|---|---|---|---|
| australian | 0, 1, 2 | 82,69 ± 3,47 | 84,94 ± 2,94 | **86,54 ± 1,67** | 87,50 ± 4,41 | **88,46 ± 3,33** | 87,18 ± 3,64 |
| blood | 1, 2 | **76,58 ± 3,82** | **76,58 ± 5,10** | 69,82 ± 0,64 | 73,87 ± 3,82 | 75,23 ± 1,91 | **76,58 ± 2,55** |
| christine | 1, 2 | 68,43 ± 2,78 | 68,30 ± 4,17 | **68,61 ± 2,52** | 70,58 ± 2,00 | **70,82 ± 2,69** | 70,52 ± 0,69 |
| higgs | 0, 1, 2 | 61,71 ± 1,51 | 64,62 ± 0,96 | **64,65 ± 0,94** | 68,17 ± 0,69 | **68,87 ± 0,83** | 68,63 ± 0,79 |
| iris | 2 | **100,00** | **100,00** | **100,00** | **100,00** | **100,00** | **100,00** |
| jasmine | 1, 2 | 77,12 ± 1,10 | 77,12 ± 2,37 | **78,91 ± 0,16** | 77,68 ± 0,32 | **78,24 ± 0,47** | 77,68 ± 0,32 |
| jsc | 1 | 71,45 | **72,61** | 70,28 | **72,01** | 70,44 | 69,09 |
| miniboone | 0, 1, 2 | 91,39 ± 0,53 | 91,55 ± 0,13 | **91,86 ± 0,21** | 92,06 ± 0,66 | 92,24 ± 0,11 | **92,26 ± 0,06** |
| mnist | 1, 2 | **95,77 ± 0,21** | 95,34 ± 0,69 | 95,20 ± 0,74 | **96,22 ± 0,16** | 95,77 ± 0,61 | 95,81 ± 0,69 |
| nomao | 0, 1, 2 | 94,74 ± 0,33 | **94,98 ± 0,11** | 94,79 ± 0,20 | 95,15 ± 0,12 | 95,18 ± 0,01 | **95,22 ± 0,13** |
| phoneme | 2 | 89,14 | **89,75** | 88,89 | 89,38 | **89,63** | 88,77 |
| segment | 0, 1, 2 | 99,13 ± 0,50 | 99,13 ± 0,50 | **99,42 ± 0,58** | 98,94 ± 0,44 | 98,84 ± 0,50 | **99,42 ± 0,58** |
| skin-seg | 0, 1, 2 | **99,60 ± 0,30** | 99,31 ± 0,23 | 99,43 ± 0,24 | 99,65 ± 0,32 | **99,80 ± 0,07** | 99,80 ± 0,04 |
| sylvine | 1, 2 | 88,31 ± 0,55 | 89,87 ± 1,65 | **90,06 ± 1,38** | 91,62 ± 1,38 | **91,88 ± 0,46** | 91,62 ± 1,56 |

**Resumo — teste médio de cada conjunto (14 conjuntos); vitórias do primeiro modo:**

- One-shot, DAB − Iterativo: 8 vitórias, 3 empates e 3 derrotas; Δ médio +0,57 pp, mediano +0,08 pp; Wilcoxon p = 0,083.
- One-shot, rDAB − Iterativo: 8 vitórias, 1 empate e 5 derrotas; Δ médio +0,17 pp, mediano +0,11 pp; Wilcoxon p = 0,455.
- One-shot, rDAB − DAB: 8 vitórias, 1 empate e 5 derrotas; Δ médio −0,40 pp, mediano +0,07 pp; Wilcoxon p = 0,787.
- Épocas, DAB − Iterativo: 10 vitórias, 1 empate e 3 derrotas; Δ médio +0,18 pp, mediano +0,22 pp; Wilcoxon p = 0,127.
- Épocas, rDAB − Iterativo: 6 vitórias, 3 empates e 5 derrotas; Δ médio −0,02 pp, mediano 0,00 pp; Wilcoxon p = 0,966.
- Épocas, rDAB − DAB: 5 vitórias, 1 empate e 8 derrotas; Δ médio −0,20 pp, mediano −0,12 pp; Wilcoxon p = 0,305.

conjunto,semente,modelo,teste Iterativo,teste DAB,teste rDAB,configurações (it / DAB / rDAB),escolhida Iterativo,escolhida DAB,escolhida rDAB
australian,0,One-shot,"79,81","81,73","84,62",18 / 30 / 30,8 / 12,2 / 8,2 / 8
australian,0,Épocas,"83,65","86,54","85,58",2268 / 600 / 600,4 / 4 · b 1 · ρ₀ 0.8 · τ 0.05 · época 2/6,2 / 8 · b 1 · ρ₀ 0.5 · τ 0.2 · época 0/6,2 / 8 · b 1 · ρ₀ 0.1 · τ 0.0 · época 0/6
australian,1,One-shot,"81,73","85,58","87,50",30 / 30 / 30,16 / 8,2 / 4,2 / 4
australian,1,Épocas,"86,54","86,54","84,62",1200 / 600 / 600,2 / 2 · b 1 · ρ₀ 0.9 · τ 0.05 · época 6/10,2 / 4 · b 1 · ρ₀ 0.9 · τ 0.2 · época 9/15,16 / 8 · b 1 · ρ₀ 0.9 · τ 0.0 · época 1/7
australian,2,One-shot,"86,54","87,50","87,50",30 / 30 / 30,8 / 16,32 / 8,8 / 16
australian,2,Épocas,"92,31","92,31","91,35",1200 / 600 / 600,2 / 2 · b 1 · ρ₀ 0.1 · τ 0.1 · época 0/4,4 / 2 · b 1 · ρ₀ 0.3 · τ 0.2 · época 3/9,4 / 8 · b 1 · ρ₀ 0.9 · τ 0.05 · época 0/6
blood,1,One-shot,"73,87","72,97","69,37",25 / 25 / 25,16 / 16,16 / 16,16 / 16
blood,1,Épocas,"71,17","73,87","74,77",1000 / 500 / 500,16 / 16 · b 2 · ρ₀ 0.7 · τ 0.05 · época 7/11,16 / 2 · b 1 · ρ₀ 0.9 · τ 0.1 · época 3/9,2 / 2 · b 1 · ρ₀ 0.9 · τ 0.0 · época 0/6
blood,2,One-shot,"79,28","80,18","70,27",25 / 25 / 25,4 / 8,8 / 8,16 / 2
blood,2,Épocas,"76,58","76,58","78,38",1000 / 500 / 500,32 / 2 · b 2 · ρ₀ 0.9 · τ 0.1 · época 7/11,32 / 4 · b 1 · ρ₀ 0.9 · τ 0.2 · época 7/13,32 / 8 · b 1 · ρ₀ 0.7 · τ 0.0 · época 0/6


### Termômetro linear — iterativo × DAB × rDAB

In [15]:
display(Markdown(modes_table("thermometer")))
modes_detail("thermometer")

| Conjunto | sementes | One-shot Iterativo | One-shot DAB | One-shot rDAB | Épocas Iterativo | Épocas DAB | Épocas rDAB |
|---|---|---|---|---|---|---|---|
| australian | 0, 1, 2 | 84,29 ± 4,84 | 85,26 ± 2,00 | **86,22 ± 0,56** | 84,29 ± 6,96 | **86,54 ± 5,35** | 85,26 ± 4,00 |
| blood | 1, 2 | **76,58 ± 0,00** | 75,23 ± 1,91 | 73,87 ± 2,55 | 76,13 ± 0,64 | 72,52 ± 5,73 | **76,58 ± 1,27** |
| christine | 0, 1, 2 | **69,41 ± 3,42** | 67,98 ± 2,02 | 68,02 ± 0,86 | 68,84 ± 2,56 | **70,52 ± 2,30** | 69,82 ± 0,43 |
| fashion-mnist | 2 | **83,61** | 83,53 | 83,46 | 84,86 | **85,22** | 84,26 |
| higgs | 0, 1, 2 | 57,85 ± 1,31 | 59,71 ± 0,97 | **59,74 ± 0,97** | **62,85 ± 1,04** | 61,50 ± 0,54 | 61,06 ± 0,59 |
| iris | 2 | 95,24 | **100,00** | **100,00** | **100,00** | 90,48 | 90,48 |
| jasmine | 1, 2 | 73,21 ± 2,53 | **76,56 ± 1,26** | 76,00 ± 0,79 | **78,68 ± 0,47** | 77,57 ± 1,42 | 76,56 ± 1,26 |
| jsc | 1 | 68,16 | **70,51** | 67,54 | **70,95** | 68,17 | 64,69 |
| miniboone | 0, 1, 2 | 82,94 ± 0,57 | 82,74 ± 0,62 | **83,12 ± 0,58** | **85,97 ± 0,79** | 84,73 ± 0,90 | 83,65 ± 2,17 |
| mnist | 1, 2 | **95,96 ± 0,09** | 95,87 ± 0,17 | 95,84 ± 0,14 | **96,34 ± 0,06** | 96,33 ± 0,06 | 96,20 ± 0,00 |
| nomao | 0, 1, 2 | 95,07 ± 0,53 | **95,31 ± 0,16** | 95,23 ± 0,19 | 95,38 ± 0,47 | **95,65 ± 0,22** | 95,39 ± 0,26 |
| phoneme | 2 | 86,30 | **86,42** | 84,94 | **87,28** | 86,17 | 84,32 |
| segment | 0, 1, 2 | 98,65 ± 0,33 | **98,84 ± 0,29** | **98,84 ± 0,29** | 98,65 ± 0,33 | **98,94 ± 0,17** | 98,84 ± 0,29 |
| skin-seg | 0, 1, 2 | 99,64 ± 0,12 | **99,68 ± 0,02** | 99,61 ± 0,07 | 99,72 ± 0,09 | 99,79 ± 0,04 | **99,82 ± 0,04** |
| sylvine | 1, 2 | 87,47 ± 0,09 | **88,70 ± 2,02** | **88,70 ± 2,02** | **91,56 ± 0,92** | 88,57 ± 0,37 | 90,97 ± 0,64 |

**Resumo — teste médio de cada conjunto (15 conjuntos); vitórias do primeiro modo:**

- One-shot, DAB − Iterativo: 10 vitórias, 0 empates e 5 derrotas; Δ médio +0,80 pp, mediano +0,19 pp; Wilcoxon p = 0,121.
- One-shot, rDAB − Iterativo: 8 vitórias, 0 empates e 7 derrotas; Δ médio +0,45 pp, mediano +0,15 pp; Wilcoxon p = 0,421.
- One-shot, rDAB − DAB: 4 vitórias, 3 empates e 8 derrotas; Δ médio −0,35 pp, mediano −0,03 pp; Wilcoxon p = 0,151.
- Épocas, DAB − Iterativo: 6 vitórias, 0 empates e 9 derrotas; Δ médio −1,25 pp, mediano −1,11 pp; Wilcoxon p = 0,169.
- Épocas, rDAB − Iterativo: 6 vitórias, 0 empates e 9 derrotas; Δ médio −1,57 pp, mediano −0,58 pp; Wilcoxon p = 0,083.
- Épocas, rDAB − DAB: 3 vitórias, 1 empate e 11 derrotas; Δ médio −0,32 pp, mediano −0,44 pp; Wilcoxon p = 0,119.

conjunto,semente,modelo,teste Iterativo,teste DAB,teste rDAB,configurações (it / DAB / rDAB),escolhida Iterativo,escolhida DAB,escolhida rDAB
australian,0,One-shot,"83,65","84,62","85,58",18 / 30 / 30,4 / 16,2 / 2,2 / 2
australian,0,Épocas,"79,81","87,50","86,54",2268 / 600 / 600,4 / 4 · b 2 · ρ₀ 0.7 · τ 0.3 · época 2/6,2 / 2 · b 1 · ρ₀ 0.1 · τ 0.05 · época 0/6,2 / 2 · b 1 · ρ₀ 0.5 · τ 0.0 · época 1/7
australian,1,One-shot,"79,81","83,65","86,54",30 / 30 / 30,16 / 16,16 / 16,2 / 4
australian,1,Épocas,"80,77","80,77","80,77",1200 / 600 / 600,16 / 16 · b 1 · ρ₀ 0.7 · τ 0.0 · época 0/4,16 / 16 · b 1 · ρ₀ 0.7 · τ 0.0 · época 0/6,16 / 16 · b 1 · ρ₀ 0.7 · τ 0.0 · época 0/6
australian,2,One-shot,"89,42","87,50","86,54",30 / 30 / 30,4 / 16,4 / 16,4 / 16
australian,2,Épocas,"92,31","91,35","88,46",1200 / 600 / 600,2 / 2 · b 1 · ρ₀ 0.3 · τ 0.1 · época 3/7,2 / 2 · b 1 · ρ₀ 0.5 · τ 0.1 · época 3/9,4 / 16 · b 1 · ρ₀ 0.7 · τ 0.0 · época 4/10
blood,1,One-shot,"76,58","76,58","75,68",25 / 25 / 25,8 / 16,16 / 2,16 / 2
blood,1,Épocas,"75,68","68,47","75,68",1000 / 500 / 500,32 / 4 · b 2 · ρ₀ 0.3 · τ 0.2 · época 0/4,16 / 8 · b 1 · ρ₀ 0.5 · τ 0.05 · época 0/6,16 / 2 · b 1 · ρ₀ 0.7 · τ 0.05 · época 0/6
blood,2,One-shot,"76,58","73,87","72,07",25 / 25 / 25,16 / 16,32 / 32,32 / 32
blood,2,Épocas,"76,58","76,58","77,48",1000 / 500 / 500,16 / 8 · b 1 · ρ₀ 0.1 · τ 0.0 · época 2/6,16 / 8 · b 1 · ρ₀ 0.1 · τ 0.05 · época 1/7,2 / 8 · b 1 · ρ₀ 0.1 · τ 0.05 · época 0/6


## DataFrames para exportar

- `long`: uma linha por conjunto, binarização e semente (todas as colunas);
- `aggregate`: uma linha por conjunto e binarização, com média e desvio entre as sementes.

Por exemplo: `long.to_csv("benchmark_por_semente.csv", index=False, decimal=",", sep=";")`.